## Gold Layer - Race Results

This notebook creates a business-ready dataset
combining drivers, races, constructors and circuits.

In [0]:
%run "/Workspace/f1-project/00-Configurations"

In [0]:
from pyspark.sql.functions import col

Read Silver Tables

In [0]:
results_df = spark.table("f1_catalog.f1_silver.results_1")

drivers_df = spark.table("f1_catalog.f1_silver.drivers_1")

constructors_df = spark.table("f1_catalog.f1_silver.constructors_1")

races_df = spark.table("f1_catalog.f1_silver.races_1")

circuits_df = spark.table("f1_catalog.f1_silver.circuits_1")

In [0]:
spark.sql("SHOW TABLES IN f1_catalog.f1_silver").show()

+---------+--------------+-----------+
| database|     tableName|isTemporary|
+---------+--------------+-----------+
|f1_silver|    circuits_1|      false|
|f1_silver|constructors_1|      false|
|f1_silver|     drivers_1|      false|
|f1_silver|       races_1|      false|
|f1_silver|     results_1|      false|
+---------+--------------+-----------+



In [0]:
drivers_df.show()

+---------+-------------+------+----+------------------+----------+-------------+--------------------+
|driver_id|   driver_ref|number|code|       driver_name|       dob|  nationality|      ingestion_date|
+---------+-------------+------+----+------------------+----------+-------------+--------------------+
|      148|       foitek|  NULL|NULL|     Gregor Foitek|1965-03-27|        Swiss|2026-03-14 11:09:...|
|      463|  chamberlain|  NULL|NULL|   Jay Chamberlain|1925-12-29|     American|2026-03-14 11:09:...|
|      471|    johnstone|  NULL|NULL|   Bruce Johnstone|1937-01-30|South African|2026-03-14 11:09:...|
|      496|   menditeguy|  NULL|NULL| Carlos Menditeguy|1914-08-10|    Argentine|2026-03-14 11:09:...|
|      833|        merhi|    98| MER|     Roberto Merhi|1991-03-22|      Spanish|2026-03-14 11:09:...|
|      243|    stommelen|  NULL|NULL|    Rolf Stommelen|1943-07-11|       German|2026-03-14 11:09:...|
|      392|       fisher|  NULL|NULL|       Mike Fisher|1943-03-13|     A

Join Results with Drivers - To get driver name and number.

In [0]:
race_results_df = results_df.join(
    drivers_df,
    results_df.driver_id == drivers_df.driver_id,
    "inner"
)

Join Results with Constructors - To get team information.

In [0]:
race_results_df = race_results_df.join(
    constructors_df,
    race_results_df.constructor_id == constructors_df.constructor_id,
    "inner"
)

Join Results with Races - To get race name and year.

In [0]:
race_results_df = race_results_df.join(
    races_df,
    race_results_df.race_id == races_df.race_id,
    "inner"
)

Join Result with Circuits - To get tracks information.

In [0]:
race_results_df = race_results_df.join(
    circuits_df,
    race_results_df.circuit_id == circuits_df.circuit_id,
    "inner"
)

Select Required Business Columns

In [0]:
race_results_final_df = race_results_df.select(
    col("race_year"),
    col("name").alias("race_name"),
    col("location").alias("circuit_location"),
    col("driver_name"),
    col("constructor_name"),
    col("grid"),
    col("position"),
    col("points")
)

In [0]:
race_results_final_df.show()

+---------+--------------------+----------------+---------------+----------------+----+--------+------+
|race_year|           race_name|circuit_location|    driver_name|constructor_name|grid|position|points|
+---------+--------------------+----------------+---------------+----------------+----+--------+------+
|     1954|Indianapolis Moto...|    Indianapolis|    Jimmy Reece|        Pankratz|   7|      17|   0.0|
|     1954|Indianapolis Moto...|    Indianapolis|   Troy Ruttman|    Kurtis Kraft|  11|       4|   1.5|
|     1954|Indianapolis Moto...|    Indianapolis|   Danny Kladis|          Bromme|  29|    NULL|   0.0|
|     1954|Indianapolis Moto...|    Indianapolis|  Jimmy Daywalt|         Nichels|   4|    NULL|   0.0|
|     1954|Indianapolis Moto...|    Indianapolis|      Sam Hanks|    Kurtis Kraft|  10|    NULL|   0.0|
|     1954|Indianapolis Moto...|    Indianapolis|   Pat O'Connor|    Kurtis Kraft|  12|    NULL|   0.0|
|     1954|Indianapolis Moto...|    Indianapolis|Chuck Stevenson

In [0]:
%sql
USE CATALOG f1_catalog

In [0]:
%sql
USE SCHEMA f1_gold

In [0]:
race_results_final_df.write.mode("overwrite").format("delta").saveAsTable('race_results')


In [0]:
%sql
select * from f1_gold.race_results

race_year,race_name,circuit_location,driver_name,constructor_name,grid,position,points
1952,Indianapolis Motor Speedway,Indianapolis,Gene Hartley,Kurtis Kraft,18,null,0.0
1952,Indianapolis Motor Speedway,Indianapolis,Jim Rigsby,Watson,26,12,0.0
1952,Indianapolis Motor Speedway,Indianapolis,Sam Hanks,Kurtis Kraft,5,3,4.0
1952,Indianapolis Motor Speedway,Indianapolis,Jimmy Reece,Kurtis Kraft,23,7,0.0
1952,Indianapolis Motor Speedway,Indianapolis,Bill Schindler,Stevens,15,14,0.0
1952,Indianapolis Motor Speedway,Indianapolis,Jim Rathmann,Kurtis Kraft,10,2,6.0
1952,Indianapolis Motor Speedway,Indianapolis,George Connor,Kurtis Kraft,14,8,0.0
1952,Indianapolis Motor Speedway,Indianapolis,Alberto Ascari,Ferrari,19,null,0.0
1952,Indianapolis Motor Speedway,Indianapolis,Andy Linden,Kurtis Kraft,2,null,0.0
1952,Indianapolis Motor Speedway,Indianapolis,Johnnie Parsons,Kurtis Kraft,31,10,0.0


Write in ADLS

In [0]:
race_results_final_df.write.mode("overwrite").format("delta").save(f"{gold_path}/race_results")